# Observed populations

This notebook reproduces Figures 3 and 4 in [Graber et al. (2024)](https://arxiv.org/abs/2312.14848).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib import rc
import matplotlib as mpl
from scipy.integrate import quad
from scipy import stats

from mlpoppyns.simulator.config_simulator import cfg
import mlpoppyns.simulator.basics.constants as const
import utilities.plot_settings

from matplotlib import rc

rc("text", usetex=True)
rc("font", family="serif")
mpl.rcParams["text.latex.preamble"] = r"\usepackage{amsmath}"

In [ ]:
SMALL_SIZE = 30
MEDIUM_SIZE = 40
BIGGER_SIZE = 60

plt.rc("font", size=SMALL_SIZE)  # controls default text sizes
plt.rc("axes", titlesize=MEDIUM_SIZE)  # fontsize of the axes title
plt.rc("axes", labelsize=MEDIUM_SIZE)  # fontsize of the x and y labels
plt.rc("xtick", labelsize=MEDIUM_SIZE)  # fontsize of the tick labels
plt.rc("ytick", labelsize=MEDIUM_SIZE)  # fontsize of the tick labels
plt.rc("legend", fontsize=SMALL_SIZE)  # legend fontsize
plt.rc("figure", titlesize=BIGGER_SIZE)  # fontsize of the figure title

## Load observed data

Load the ATNF Pulsar Catalogue v1.69.

In [ ]:
# Read the full ATNF catalog.csv file. Binary pulsars are excluded.
df_atnf = pd.read_csv(
    "../../data/observations/atnf_full_nobinary_25-04-2023.csv",
    delimiter=";",
    header=[0, 1],
)
df_atnf.head()

In [ ]:
df_atnf = df_atnf.drop(
    columns=[
        "#",
        "PMRA",
        "PMDEC",
        "PX",
        "POSEPOCH",
        "RAJD",
        "DECJD",
        "DM",
        "W50",
        "W10",
        "TAU_SC",
        "S400",
        "S2000",
        "DIST",
        "DIST_DM",
        "ZZ",
        "XX",
        "YY",
        "PSR",
        "Unnamed: 27_level_0",
    ],
    level=0,
)

len(df_atnf)

In [ ]:
# Select only those with measured period values.
df_atnf = df_atnf[~df_atnf["P0"]["(s)"].isin(["NAN"])]

# Remove those objects that are in globular clusters or in the Magellanic Clouds.
discard = [
    "EXGAL:SMC",
    "EXGAL:LMC",
    "GC:47Tuc",
    "GC:M3",
    "GC:M5",
    "GC:M13",
    "GC:NGC6440",
    "GC:Ter5",
    "GC:NGC6441",
    "GC:NGC6517",
    "GC:NGC6522",
    "GC:NGC6624",
    "GC:M28(NGC6626)",
    "GC:NGC6652",
    "GC:M22(NGC6656)",
    "GC:NGC6752",
    "GC:NGC6760",
    "GC:M15",
    "GC:M30",
]
df_atnf = df_atnf[
    ~df_atnf["ASSOC"]["Unnamed: 24_level_1"].str.match("|".join(discard))
]

len(df_atnf)

In [ ]:
# Select only isolated non-recycled neutron stars through filters with P > 0.01 and Pdot > 1e-19 (for those with measured values).
df_atnf = df_atnf[df_atnf["P0"]["(s)"].to_numpy().astype(np.float64) > 0.01]

len(df_atnf)

Note: For the purpose of modelling the observed isolated population of radio pulsars, in the following, we count those objects with period derivatives larger than $\dot{P} > 10^{-19} s/s$ or those with no measured period derivatives. The latter are likely isolated in nature due to the fact that only a small fraction of ATNF pulsars with known $\dot{P}$ actually have been recycled and attain $\dot{P} < 10^{-19} s/s$. As a result, the following number counts are slighlty larger than the populations used to produce our period period-derivative maps for the simulation-based inference approach.

In [ ]:
df_atnf = df_atnf[
    (df_atnf["P1"]["(s/s)"].to_numpy().astype(np.float64) > 1.0e-19)
    | (df_atnf["P1"]["(s/s)"].isin(["NAN"]))
]

In [ ]:
# Parkes Multibeam Pulsar Survey (PMPS) database.
df_atnf_pmps = df_atnf[
    df_atnf["SURVEY"]["Unnamed: 25_level_1"].str.contains("pksmb")
]

l_pmps_obs = df_atnf_pmps["Gl"]["(deg)"].to_numpy().astype(np.float64)
b_pmps_obs = df_atnf_pmps["Gb"]["(deg)"].to_numpy().astype(np.float64)
P_pmps_obs = df_atnf_pmps["P0"]["(s)"].to_numpy().astype(np.float64)
Pdot_pmps_obs = df_atnf_pmps["P1"]["(s/s)"].to_numpy().astype(np.float64)
S1400_pmps_obs = df_atnf_pmps["S1400"]["(mJy)"].to_numpy().astype(np.float64)

# Convert galactic latitude in the range [-180., 180].
l_pmps_obs[(l_pmps_obs > 180.0) & (l_pmps_obs < 360.0)] = (
    l_pmps_obs[(l_pmps_obs > 180.0) & (l_pmps_obs < 360.0)] - 360.0
)

# Select only pulsars falling in the PMPS sky coverage where completness is above 90%.
cond = (l_pmps_obs > -100.0) & (l_pmps_obs < 50.0) & (np.abs(b_pmps_obs) < 5.0)

l_pmps_obs = l_pmps_obs[cond]
b_pmps_obs = b_pmps_obs[cond]
P_pmps_obs = P_pmps_obs[cond]
Pdot_pmps_obs = Pdot_pmps_obs[cond]
S1400_pmps_obs = S1400_pmps_obs[cond]

number_pmps = len(l_pmps_obs)

print(number_pmps)

In [ ]:
# Swinburne Intermediate-latitude Pulsar Survey (SMPS) database.
df_atnf_smps = df_atnf[
    df_atnf["SURVEY"]["Unnamed: 25_level_1"].str.contains("pkssw")
]

l_smps_obs = df_atnf_smps["Gl"]["(deg)"].to_numpy().astype(np.float64)
b_smps_obs = df_atnf_smps["Gb"]["(deg)"].to_numpy().astype(np.float64)
P_smps_obs = df_atnf_smps["P0"]["(s)"].to_numpy().astype(np.float64)
Pdot_smps_obs = df_atnf_smps["P1"]["(s/s)"].to_numpy().astype(np.float64)
S1400_smps_obs = df_atnf_smps["S1400"]["(mJy)"].to_numpy().astype(np.float64)

# Convert galactic latitude in the range [-180., 180].
l_smps_obs[(l_smps_obs > 180.0) & (l_smps_obs < 360.0)] = (
    l_smps_obs[(l_smps_obs > 180.0) & (l_smps_obs < 360.0)] - 360.0
)

# Selection only pulsars falling in the SMPS sky coverage where completness is above 90%.
cond = (l_smps_obs > -100.0) & (l_smps_obs < 50.0)

l_smps_obs = l_smps_obs[cond]
b_smps_obs = b_smps_obs[cond]
P_smps_obs = P_smps_obs[cond]
Pdot_smps_obs = Pdot_smps_obs[cond]
S1400_smps_obs = S1400_smps_obs[cond]

l_all_obs = np.concatenate((l_pmps_obs, l_smps_obs))
b_all_obs = np.concatenate((b_pmps_obs, b_smps_obs))
P_all_obs = np.concatenate((P_pmps_obs, P_smps_obs))
Pdot_all_obs = np.concatenate((Pdot_pmps_obs, Pdot_smps_obs))
S1400_all_obs = np.concatenate((S1400_pmps_obs, S1400_smps_obs))

number_smps = len(l_smps_obs)
print(number_smps)

In [ ]:
# Low- and mid-latitude High Time Resolution Universe (HTRU) database.
df_atnf_htru = df_atnf[
    df_atnf["SURVEY"]["Unnamed: 25_level_1"].str.contains("htru_pks")
]

l_htru_obs = df_atnf_htru["Gl"]["(deg)"].to_numpy().astype(np.float64)
b_htru_obs = df_atnf_htru["Gb"]["(deg)"].to_numpy().astype(np.float64)
P_htru_obs = df_atnf_htru["P0"]["(s)"].to_numpy().astype(np.float64)
Pdot_htru_obs = df_atnf_htru["P1"]["(s/s)"].to_numpy().astype(np.float64)
S1400_htru_obs = df_atnf_htru["S1400"]["(mJy)"].to_numpy().astype(np.float64)

# Convert galactic latitude in the range [-180., 180].
l_htru_obs[(l_htru_obs > 180.0) & (l_htru_obs < 360.0)] = (
    l_htru_obs[(l_htru_obs > 180.0) & (l_htru_obs < 360.0)] - 360.0
)

# Selection only pulsars falling in the HTRU sky coverage where completness is above 90%.
cond = (
    (l_htru_obs > -120.0) & (l_htru_obs < 30.0) & (np.abs(b_htru_obs) < 15.0)
)

l_htru_obs = l_htru_obs[cond]
b_htru_obs = b_htru_obs[cond]
P_htru_obs = P_htru_obs[cond]
Pdot_htru_obs = Pdot_htru_obs[cond]
S1400_htru_obs = S1400_htru_obs[cond]

l_all_obs = np.concatenate((l_pmps_obs, l_smps_obs, l_htru_obs))
b_all_obs = np.concatenate((b_pmps_obs, b_smps_obs, b_htru_obs))
P_all_obs = np.concatenate((P_pmps_obs, P_smps_obs, P_htru_obs))
Pdot_all_obs = np.concatenate((Pdot_pmps_obs, Pdot_smps_obs, Pdot_htru_obs))
S1400_all_obs = np.concatenate(
    (S1400_pmps_obs, S1400_smps_obs, S1400_htru_obs)
)

number_htru = len(l_htru_obs)
print(number_htru)

In [ ]:
print(f"Number of pulsars detected by PMPS: {number_pmps}")
print(f"Number of pulsars detected by SMPS: {number_smps}")
print(f"Number of pulsars detected by HTRU: {number_htru}")

In [ ]:
print(
    f"Number of pulsars detected by PMPS without Pdot measurement: {np.isnan(Pdot_pmps_obs).sum()}"
)
print(
    f"Number of pulsars detected by SMPS multibeam without Pdot measurement: {np.isnan(Pdot_smps_obs).sum()}"
)
print(
    f"Number of pulsars detected by HTRU without Pdot measurement: {np.isnan(Pdot_htru_obs).sum()}"
)

In [ ]:
print(
    f"Number of pulsars detected by PMPS without S1400 measurement: {np.isnan(S1400_pmps_obs).sum()}"
)
print(
    f"Number of pulsars detected by SMPS multibeam without S1400 measurement: {np.isnan(S1400_smps_obs).sum()}"
)
print(
    f"Number of pulsars detected by HTRU without S1400 measurement: {np.isnan(S1400_htru_obs).sum()}"
)

## Plotting the observations

Sky positions.

In [ ]:
colors = ["#FFAC1C", "dodgerblue", "#440154"]

In [ ]:
fig, ax = plt.subplots(figsize=(15, 10))

plt.plot(
    l_pmps_obs,
    b_pmps_obs,
    linestyle="None",
    marker="o",
    color=colors[0],
    markersize=8,
    alpha=1,
    rasterized=True,
    label=r"Observed PMPS",
)
ax.plot(
    l_smps_obs,
    b_smps_obs,
    linestyle="None",
    marker="o",
    color=colors[1],
    markersize=8,
    alpha=1,
    rasterized=True,
    label=r"Observed SMPS",
)
ax.plot(
    l_htru_obs,
    b_htru_obs,
    linestyle="None",
    marker="o",
    # fillstyle="none",
    color=colors[2],
    markersize=8,
    alpha=0.3,
    rasterized=True,
    label=r"Observed HTRU",
)

ax.set_xlim(-130.0, 80.0)
ax.set_ylim(-50.0, 50.0)
ax.set_xlabel(r"Galactic longitude $l$ [deg]")
ax.set_ylabel(r"Galactic latitude $b$ [deg]")
ax.legend(frameon=True, loc="best")
ax.grid()

plt.tight_layout()
plt.savefig(
    "../../paper_plots/graber_etal_2024/plots/observed_coord.pdf", dpi=300, bbox_inches="tight"
)
plt.show()

Edot lines.

In [ ]:
I_NS = 1.36e45
R_NS = 1.1e6
c = 2.998e10

Pdot_Edot_lines = np.zeros((6, 51))

Edot_log = np.linspace(28, 38, 6)
print(Edot_log)

In [ ]:
P_log = np.linspace(-3, 2, 51)

In [ ]:
for i in range(len(Edot_log)):
    Pdot_Edot_lines[i] = (
        10 ** Edot_log[i] * (10**P_log) ** 3 / (4 * np.pi**2 * I_NS)
    )

B lines.

In [ ]:
Pdot_B_lines = np.zeros((5, 51))

B_log = np.linspace(10, 14, 5)
print(B_log)

In [ ]:
for i in range(len(B_log)):
    Pdot_B_lines[i] = (
        np.pi**2
        * (10 ** B_log[i]) ** 2
        * (R_NS**6)
        / (I_NS * 10**P_log * c**3)
    )

PPdot diagrams.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 10))

for i in range(len(Edot_log)):
    ax.plot(
        10**P_log,
        Pdot_Edot_lines[i],
        linestyle="-",
        color="gray",
        alpha=0.5,
        rasterized=True,
    )
for i in range(len(B_log)):
    ax.plot(
        10**P_log,
        Pdot_B_lines[i],
        linestyle="-",
        color="gray",
        alpha=0.5,
        rasterized=True,
    )
ax.plot(
    P_pmps_obs,
    Pdot_pmps_obs,
    linestyle="None",
    marker="o",
    color=colors[0],
    markersize=9,
    alpha=1.0,
    rasterized=True,
    label=r"Observed PMPS",
)
ax.plot(
    P_smps_obs,
    Pdot_smps_obs,
    linestyle="None",
    marker="o",
    color=colors[1],
    markersize=9,
    alpha=1.0,
    rasterized=True,
    label=r"Observed SMPS",
)
ax.plot(
    P_htru_obs,
    Pdot_htru_obs,
    linestyle="None",
    marker="o",
    color=colors[2],
    markersize=9,
    alpha=0.3,
    rasterized=True,
    label=r"Observed HTRU",
)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(1.0e-3, 100.0)
ax.set_ylim(1.0e-21, 1.0e-9)

ax.text(
    0.41,
    0.015,
    r"$10^{28} \, {\rm erg} \, {\rm s}^{-1}$",
    rotation=50,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)
ax.text(
    0.275,
    0.015,
    r"$10^{30} \, {\rm erg} \, {\rm s}^{-1}$",
    rotation=50,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)
ax.text(
    0.143,
    0.015,
    r"$10^{32} \, {\rm erg} \, {\rm s}^{-1}$",
    rotation=50,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)
ax.text(
    0.01,
    0.015,
    r"$10^{34} \, {\rm erg} \, {\rm s}^{-1}$",
    rotation=50,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)
ax.text(
    0.01,
    0.181,
    r"$10^{36} \, {\rm erg} \, {\rm s}^{-1}$",
    rotation=50,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)
ax.text(
    0.01,
    0.347,
    r"$10^{38} \, {\rm erg} \, {\rm s}^{-1}$",
    rotation=50,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)

ax.text(
    0.89,
    0.005,
    r"$10^{10} \, {\rm G}$",
    rotation=-23,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)
ax.text(
    0.89,
    0.172,
    r"$10^{11} \, {\rm G}$",
    rotation=-23,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)
ax.text(
    0.89,
    0.340,
    r"$10^{12} \, {\rm G}$",
    rotation=-23,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)
ax.text(
    0.89,
    0.506,
    r"$10^{13} \, {\rm G}$",
    rotation=-23,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)
ax.text(
    0.89,
    0.672,
    r"$10^{14} \, {\rm G}$",
    rotation=-23,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)

plt.xlabel(r"Period $P$ [s]")
plt.ylabel(r"Period derivative $\dot{P}$ [s$\,{\rm s}^{-1}$]")
ax.legend(frameon=True, loc=2)

plt.tight_layout()
plt.savefig(
    "../../paper_plots/graber_etal_2024/plots/observed_ppdot.pdf", dpi=300, bbox_inches="tight"
)
plt.show()

Plotting the observed fluxes at 1.4 GHz measured in mJy.

In [ ]:
print(min(np.log10(S1400_pmps_obs)), np.log10(max(S1400_pmps_obs)))

In [ ]:
bin_edges = np.arange(-2, 3.5, 0.3)
print(bin_edges)

In [ ]:
# New X-axis values.
x = np.linspace(-2, 3.5, 1000)

# Remove NaNs and estimate the PDF using a Gaussian kernel.
S1400_pmps_obs = S1400_pmps_obs[~np.isnan(S1400_pmps_obs)]
kde_pmps_obs = stats.gaussian_kde(np.log10(S1400_pmps_obs))

S1400_smps_obs = S1400_smps_obs[~np.isnan(S1400_smps_obs)]
kde_smps_obs = stats.gaussian_kde(np.log10(S1400_smps_obs))

S1400_htru_obs = S1400_htru_obs[~np.isnan(S1400_htru_obs)]
kde_htru_obs = stats.gaussian_kde(np.log10(S1400_htru_obs))

In [ ]:
fig, ax = plt.subplots(figsize=(15, 10))

# Histograms.
ax.hist(
    np.log10(S1400_pmps_obs),
    bin_edges,
    align="mid",
    density=True,
    color=colors[0],
    alpha=0.5,
    histtype="step",
    lw=4,
    label=r"Observed PMPS",
)
ax.hist(
    np.log10(S1400_smps_obs),
    bin_edges,
    align="mid",
    density=True,
    color=colors[1],
    alpha=0.5,
    histtype="step",
    lw=4,
    label=r"Observed SMPS",
)
ax.hist(
    np.log10(S1400_htru_obs),
    bin_edges,
    align="mid",
    density=True,
    color=colors[2],
    alpha=0.5,
    histtype="step",
    lw=4,
    label=r"Observed HTRU",
)

# KDE density plots.
ax.plot(
    x,
    kde_pmps_obs(x),
    color=colors[0],
    alpha=1,
    linewidth=5,
    linestyle="--",
    label=r"KDE observed PMPS",
)
ax.plot(
    x,
    kde_smps_obs(x),
    color=colors[1],
    alpha=1,
    linewidth=5,
    linestyle="--",
    label=r"KDE observed SMPS",
)
ax.plot(
    x,
    kde_htru_obs(x),
    color=colors[2],
    alpha=1,
    linewidth=5,
    linestyle="--",
    label=r"KDE observed HTRU",
)

ax.set_xlim(-2, 3.5)
ax.set_ylim(0.0, 1)
ax.set_xlabel(r"Mean flux density log$_{10}$ $S_{{\rm mean}, 1400}$ [mJy]")
ax.set_ylabel(r"Normalized pulsar count")
ax.legend(frameon=True, loc="best")

plt.tight_layout()
plt.savefig(
    "../../paper_plots/graber_etal_2024/plots/flux_hist_observed.pdf",
    dpi=300,
    bbox_inches="tight",
)
plt.show()